In [ ]:
import aisuite as ai
import json
import re

client = ai.Client(
    {
        "openai":{
            "timeout": 100,
            "api_key": ""
        }   
    }
)

def generate_pp(client: ai.Client, question: str, file_path: str, schema: str, model: str) -> str:
    
    prompt = f"""
    You are a Python and Pandas assistant. Given the schema of a DataFrame and the user's question, write a Python code using pandas to process the DataFrame.

    Schema:
    {schema}

    User question:
    {question}

    file_path: 
    {file_path}

    Store the output in a result variable. Do not print the result. 
    
    Respond with the python only.
    Return your answer *strictly* in this format:

    <execute_python>
    # valid python code here
    </execute_python>

    Do not add explanations, only the tags and the code.
    """
    print(prompt)

    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=1,
    )
    return response.choices[0].message.content.strip()

In [17]:
schema = """
id (INTEGER)
product_id (INTEGER)
product_name (STRING)
brand (STRING)
category (STRING)
color (STRING)
action (STRING)
qty_delta (INTEGER)
unit_price (FLOAT)
notes (TEXT)
ts (DATETIME)
"""

model = "openai:gpt-4o-mini"

question = "Compute total income for Red color protucts for sale action"

file_path = "inventory_events_100.csv"

In [24]:
script = generate_pp(client, question, file_path, schema, model)



    You are a Python and Pandas assistant. Given the schema of a DataFrame and the user's question, write a Python code using pandas to process the DataFrame.

    Schema:
    
id (INTEGER)
product_id (INTEGER)
product_name (STRING)
brand (STRING)
category (STRING)
color (STRING)
action (STRING)
qty_delta (INTEGER)
unit_price (FLOAT)
notes (TEXT)
ts (DATETIME)


    User question:
    Compute total income for Red color protucts for sale action

    file_path: 
    inventory_events_100.csv

    Store the output in a result variable. Do not print the result. 

    Respond with the python only.
    Return your answer *strictly* in this format:

    <execute_python>
    # valid python code here
    </execute_python>

    Do not add explanations, only the tags and the code.
    


In [25]:
print(script)


<execute_python>
import pandas as pd

# Load the DataFrame from the CSV file
df = pd.read_csv('inventory_events_100.csv')

# Filter for Red color products and sale action
filtered_df = df[(df['color'] == 'Red') & (df['action'] == 'sale')]

# Compute total income
total_income = (filtered_df['qty_delta'] * filtered_df['unit_price']).sum()

# Store the output in a result variable
result = total_income
</execute_python>


In [29]:
match = re.search(r"<execute_python>([\s\S]*?)</execute_python>", script)
ns = {} 
if match:
    #print("Found Python code:")
    script_filtered = match.group(1).strip()
    #print(script_filtered)
    #print("--"*10)
    exec(script_filtered, ns)  # Execute the generated script
    output = ns["result"]
    #print(output)
else:
    print("No Python code found in the response.")

In [31]:
output


np.float64(-6942.29)

## Improve with feedback

In [33]:
def refine_script_external_feedback(
    question: str,
    script: str,
    output: str,
    schema: str,
    model: str,
) -> tuple[str, str]:
    """
    Reflect on whether a script's *shown output* answers the question,
    and propose an improved version if needed.
    Returns (feedback, refined_code).
    """
    prompt = f"""
You are a python (pandas) reviewer and refiner.

User asked:
{question}

Original python script:
{script}

Data Frame Columns:
{schema}

Python script OUTPUT:
{output}

Step 1: Briefly evaluate if the python script OUTPUT fully answers the user's question.
Step 2: If improvement is needed, provide a refined python script.
If the original python script is already correct, return it unchanged.

Return STRICT JSON with two fields:
{{
  "feedback": "<1-3 sentences explaining the gap or confirming correctness. DO not include code here>",
  "refined_pypandas": "<final python script to run> Respond with the python only.
                   Return your answer *strictly* in this format:
                     <execute_python>
                     # valid python code here
                     </execute_python>"
}}
"""
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=1,
    )

    content = response.choices[0].message.content

    try:
        obj = json.loads(content)
        feedback = str(obj.get("feedback", "")).strip()
        refined_script = str(obj.get("refined_pypandas", script)).strip()
        if not refined_script:
            refined_script = script
    except Exception:
        # Fallback if model doesn't return valid JSON
        feedback = content.strip()
        refined_script = script

    return feedback, refined_script

In [34]:
model = "openai:gpt-5.6-sol"

In [36]:
output

np.float64(-6942.29)

In [37]:
feedback, refined_script = refine_script_external_feedback(
    question,
    script,
    output,
    schema,
    model,
)

In [38]:
print(feedback)
print("=="*10)
print(refined_script)

The filtering is correct, but sale quantities are recorded as negative inventory changes, causing the reported income to be negative. Negate qty_delta when calculating sales income.
<execute_python>
import pandas as pd

# Load the DataFrame from the CSV file
df = pd.read_csv('inventory_events_100.csv')

# Filter for Red products sold
filtered_df = df[(df['color'] == 'Red') & (df['action'] == 'sale')]

# Convert negative inventory changes into positive quantities sold
result = ((-filtered_df['qty_delta']) * filtered_df['unit_price']).sum()
</execute_python>


In [41]:
print(script)
print(10*"---")
print(refined_script)

<execute_python>
import pandas as pd

# Load the DataFrame from the CSV file
df = pd.read_csv('inventory_events_100.csv')

# Filter for Red color products and sale action
filtered_df = df[(df['color'] == 'Red') & (df['action'] == 'sale')]

# Compute total income
total_income = (filtered_df['qty_delta'] * filtered_df['unit_price']).sum()

# Store the output in a result variable
result = total_income
</execute_python>
------------------------------
<execute_python>
import pandas as pd

# Load the DataFrame from the CSV file
df = pd.read_csv('inventory_events_100.csv')

# Filter for Red products sold
filtered_df = df[(df['color'] == 'Red') & (df['action'] == 'sale')]

# Convert negative inventory changes into positive quantities sold
result = ((-filtered_df['qty_delta']) * filtered_df['unit_price']).sum()
</execute_python>


In [42]:
match = re.search(r"<execute_python>([\s\S]*?)</execute_python>", refined_script)
ns = {} 
if match:
    #print("Found Python code:")
    script_filtered = match.group(1).strip()
    #print(script_filtered)
    #print("--"*10)
    exec(script_filtered, ns)  # Execute the generated script
    output = ns["result"]
    #print(output)
else:
    print("No Python code found in the response.")

In [44]:
output

np.float64(6942.29)